# Model aggregation report and correlations between best models

The previously identified best-performing models — Random Forest, GraphConv GNN, and SAGE GNN — were retrained on the full available dataset (57k samples).

Based on these results, an ensemble model was constructed by aggregating individual model predictions using either the mean or a linear regression. The rationale was to assess whether combining the outputs of independent models could yield improved predictive performance.

First model correlations are shown:

In [ ]:
# Model correlations plots
import numpy as np

# Load best models predictions
mod1 = np.load('/home/mriveraceron/glv-research/updated_results/SAGE_ss/ss_5/prediction_values.npz')['mp'].flatten()            # SAGE
mod2 = np.load('/home/mriveraceron/glv-research/updated_results/GraphConv_ss/ss_5/prediction_values.npz')['mp'].flatten()       # GraphConv
mod3 = np.load('/home/mriveraceron/glv-research/updated_results/RF_hpo/tree_params/rf_900_metrics.npz')['values_pred'].flatten()   # Random Forest


In [ ]:
# Generate correlations
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import pearsonr, spearmanr

# Add the font directory directly
import matplotlib.font_manager as fm
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman.ttf')
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman Bold.ttf')
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman Italic.ttf')
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman Bold Italic.ttf')

plt.rcParams.update({
    'font.family':  'serif',
    'font.serif':   'Times New Roman',
    'font.size':    12,                  # sets the global base size
    'axes.facecolor':   'white',        # ← add this
    'figure.facecolor': 'white',        # ← and this (covers fig too)
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.8,
    'grid.alpha':       0.7,
})

def corr_plt(data_x, data_y, ax_title, ax, label, x_axis, y_axis):
    fig = ax.get_figure()
    # --- Config ---
    LIMS        = [-0.02, 1.05]
    TICK_MAJOR  = 0.2
    TICK_MINOR  = 0.05
    FMT         = '%.1f'
    # --- Data & correlations ---
    r_p, p_val_p = pearsonr(data_x, data_y)
    print(f'Pearson r: {r_p:.3f}, p-value: {p_val_p:.2e}')
    r_s, p_val_s = spearmanr(data_x, data_y)
    print(f'Spearman ρ: {r_s:.3f}, p-value: {p_val_s:.2e}')
    x, y   = data_x, data_y
    # --- Figure ---
    # Hexbin
    hb = ax.hexbin(x, y, gridsize=45, cmap='YlOrRd', mincnt=1, bins='log', linewidths=0.15, edgecolors='none')
    # Colorbar
    cb = fig.colorbar(hb, ax=ax, pad=0.03, shrink=0.82, aspect=22)
    cb.set_label('Recuento (escala log)', fontsize=11, labelpad=8)
    cb.ax.tick_params(labelsize=10, direction='in')
    cb.outline.set_linewidth(0.8)
    # Identity line
    ax.plot(LIMS, LIMS, color='0.3', linewidth=1.2, linestyle='--', alpha=0.85, zorder=5, label='Correlación perfecta')
    # Labels & titles
    ax.set_xlabel(x_axis, labelpad=8)
    ax.set_ylabel(y_axis, labelpad=8)
    ax.set_xlim(LIMS)
    ax.set_ylim(LIMS)
    ax.set_title(ax_title, fontsize=11, color='0.4', pad=6)
    # Ticks (both axes)
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_locator(ticker.MultipleLocator(TICK_MAJOR))
        axis.set_minor_locator(ticker.MultipleLocator(TICK_MINOR))
        axis.set_major_formatter(ticker.FormatStrFormatter(FMT))
    # Correlation annotation
    ax.text(
        0.05, 0.95,
        f'$r$ = {r_p.item():.3f}$\;$(Pearson)\n$\\rho$ = {r_s.item():.3f}$\;$(Spearman)',
        transform=ax.transAxes, ha='left', va='top',
        fontsize=10, zorder=10,
        bbox=dict(boxstyle='round,pad=0.5,rounding_size=0.4',facecolor='white', edgecolor='0.75', linewidth=0.8, alpha=0.9),
    )
    # Panel label
    ax.text(-0.1, 1.05, label, transform=ax.transAxes, fontsize=14, fontweight='bold', va='top', ha='left')
    # Legend & spines
    ax.legend(loc='lower right', fontsize=10, framealpha=0.9, edgecolor='0.75')
    ax.spines[['top', 'right']].set_visible(False)


plot_titles = {
    'Correlaciones GraphConv-SAGE': (mod2, mod1, 'Keystoneness predicho - GraphConv', 'Keystoneness predicho - SAGE'),
    'Correlaciones GraphConv-RF':   (mod2, mod3, 'Keystoneness predicho - GraphConv', 'Keystoneness predicho - Random Forest'),
    'Correlaciones RF-SAGE':        (mod3, mod1, 'Keystoneness predicho - Random Forest', 'Keystoneness predicho - SAGE')
}


plt.clf()
plt.close('all')
fig, axes = plt.subplots(ncols=3, figsize=(21, 7))  # ← 3 columns, wider figure
plot_id = ['A', 'B', 'C']
fig.suptitle('Figura 16: Correlaciones entre los modelos', fontsize=12, fontweight='bold')

# for method, ax, label in zip(plot_titles, axes.flat, plot_id):
method = next(iter(plot_titles))
data_x=plot_titles[method][0]
data_y=plot_titles[method][1]
ax_title=method
ax=axes.flat[0]
label=plot_id[0]
x_axis =plot_titles[method][2]
y_axis=plot_titles[method][3]

for method, ax, label in zip(plot_titles, axes.flat, plot_id):
    corr_plt(data_x=plot_titles[method][0], data_y=plot_titles[method][1], ax_title=method, ax=ax, label=label, x_axis =plot_titles[method][2], y_axis=plot_titles[method][3])


plt.tight_layout()  # ← prevents title/label overlap
save_path = '/home/mriveraceron/glv-research/thesis_plots/all_model_corr.png'
plt.savefig(save_path, bbox_inches='tight', dpi=300)
plt.show()
print(f'>> Image saved at: {save_path}')


## Caption

Figura 1. Correlación por pares entre los tres modelos con mejor desempeño. Todos los modelos fueron entrenados con el mismo conjunto de 57,000 muestras y evaluados en un conjunto de prueba de 9,700 muestras no vistas durante el entrenamiento. El color de cada hexágono indica la densidad local de datos en escala logarítmica, donde el rojo denota la mayor concentración de predicciones y el amarillo la menor. La diagonal punteada representa la correlación perfecta. (A) GraphConv vs. SAGE. (B) GraphConv vs. Random Forest (RF). (C) SAGE vs. Random Forest (RF).

## Interpretación

Los tres modelos presentan correlaciones moderadas entre sí, lo que indica que no están aprendiendo las mismas representaciones y que la importancia asignada a las variables de entrada difiere entre arquitecturas. Destaca que GraphConv y Random Forest (RF) presentan la menor correlación entre todos los pares evaluados. Esto era esperado, dado que GraphConv es el único modelo que integra los pesos de las aristas (*edge weights*) en su formulación, lo que sugiere que dicha incorporación contribuye directamente a mejorar el desempeño predictivo.

Por otro lado, los pares GraphConv-SAGE y SAGE-RF muestran correlaciones similares entre sí. Esto podría explicarse por el hecho de que ambas arquitecturas GNN reciben las mismas variables de entrada, pero difieren en la importancia que le asignan a cada una, lo que produce predicciones parcialmente distintas a pesar de compartir la misma información.


# Model Ensemble

Having established the pairwise correlations between models, we explored whether combining them through an ensemble approach could outperform the best individual model. Specifically, we evaluated two aggregation strategies: a simple mean of the model outputs and a linear regression to estimate the optimal combination coefficients.

In [ ]:
# Model ensemble
mp = np.load('/home/mriveraceron/glv-research/updated_results/GraphConv_ss/ss_5/prediction_values.npz')['mp'].flatten()       # GraphConv
mt = np.load('/home/mriveraceron/glv-research/updated_results/GraphConv_ss/ss_5/prediction_values.npz')['mt'].flatten()       # Expected
data = np.load('/home/mriveraceron/glv-research/updated_results/aggr_preds.npz')
plot_titles = {
    'Ensamble con regresión': (data['aggr_regression'], mt, 'Keystoneness predicho', 'Keystoneness observado'),
    'Ensamble con promedios':   (data['aggr_mean'], mt, 'Keystoneness predicho', 'Keystoneness observado'),
    'Predicción con GraphConv':   (mp, mt, 'Keystoneness predicho', 'Keystoneness observado')
}

plt.clf()
plt.close('all')
fig, axes = plt.subplots(ncols=3, figsize=(21, 7))  # ← 3 columns, wider figure
plot_id = ['A', 'B', 'C']
fig.suptitle('Figura 17: Comparación de correlaciones por método de ensamble: Modelo agregado GraphConv-SAGE', fontsize=12, fontweight='bold')
for method, ax, label in zip(plot_titles, axes.flat, plot_id):
    corr_plt(data_x=plot_titles[method][0], data_y=plot_titles[method][1], ax_title=method, ax=ax, label=label, x_axis =plot_titles[method][2], y_axis=plot_titles[method][3])


plt.tight_layout()  # ← prevents title/label overlap
save_path = '/home/mriveraceron/glv-research/thesis_plots/all_model_ensemble_corr.png'
plt.savefig(save_path, bbox_inches='tight', dpi=300)
plt.show()
print(f'>> Image saved at: {save_path}')

## Caption

Figura 2. Correlación entre los modelos ensamblados y el mejor modelo individual de referencia. Todos los modelos fueron entrenados con el mismo conjunto de 57,000 muestras y evaluados en un conjunto de prueba de 9,700 muestras no vistas durante el entrenamiento. El color de cada 
hexágono indica la densidad local de datos en escala logarítmica, donde el rojo denota la mayor concentración de predicciones y el amarillo la menor. La diagonal punteada representa la correlación perfecta. (A) Ensamble por regresión lineal para estimar los coeficientes óptimos 
de combinación. (B) Ensamble mediante el promedio simple de las predicciones de ambos modelos. (C) Mejor modelo GNN individual sin ensamble, utilizado como línea base de comparación.

## Interpretación

El ensamble de modelos no mejoró el desempeño predictivo; por el contrario, introdujo una reducción en las predicciones. Esto sugiere que la combinación de modelos incorpora ruido adicional que desplaza las predicciones fuera de los valores esperados, penalizando el desempeño en lugar de mejorarlo.

No obstante, este resultado no descarta el ensamble como estrategia viable. Es posible que arquitecturas con mayor generalización produzcan predicciones más complementarias entre sí, lo que podría favorecer una combinación más efectiva y eventualmente superar el desempeño del mejor modelo individual.

In [ ]:
# Confussion matrix
# For confussion matrix we will require to compute the number of nodes
import torch
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

# Count the number of nodes
eval_dir = Path('/home/mriveraceron/glv-research/data_null/efa83c9fafa0_eval')
data_tensor = [item for f in sorted(eval_dir.glob('*.pt')) for item in torch.load(f, weights_only= False)]
# Repeat graph index i for every node in graph i
graph_id = np.concatenate([np.full(s.x.shape[0], i) for i, s in enumerate(data_tensor)])
graph_nodes = np.array([s.x.shape[0] for s in data_tensor])
graph_nodes.sum()

# Generate the confussion matrix
idxp_graphconv = np.load('/home/mriveraceron/glv-research/updated_results/GraphConv_ss/ss_5/prediction_values.npz')['idxp'].flatten()       # GraphConv
idxt = np.load('/home/mriveraceron/glv-research/updated_results/GraphConv_ss/ss_5/prediction_values.npz')['idxt'].flatten()       # Expected
idxp_regression = pd.Series(data['aggr_regression']).groupby(graph_id).idxmax().values
idxp_mean = pd.Series(data['aggr_mean']).groupby(graph_id).idxmax().values

# Compute mask once, reuse for tp and fp
def compute_cm(pred_idx, true_idx):
    correct = true_idx == pred_idx
    tp = correct.sum()
    fp = len(true_idx) - tp          # avoids a second full pass over the array
    fn = fp
    tn = graph_nodes.sum() - len(graph_nodes) - fp   # vectorized sum(nodes - 1)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    ppv      = tp / (tp + fp)
    print(f'Model aggregation | Accuracy: {accuracy:.4f} | PPV: {ppv:.4f} | By chance: {1/graph_nodes.mean():.4f}')
    df_cm = pd.DataFrame(
    [[tp, fp], [fn, tn]],
    index   = ['Expected_Positive', 'Expected_Negative'],
    columns = ['Predicted_Positive', 'Predicted_Negative']
    )
    # print(df_cm)
    display(df_cm.style.set_table_styles([{'selector': '', 'props': [('margin', '0 auto')]}]))
    print('\n')

compute_cm(pred_idx=idxp_regression, true_idx=idxt)
compute_cm(pred_idx=idxp_mean, true_idx=idxt)
compute_cm(pred_idx=idxp_graphconv, true_idx=idxt)
